# Strategy — shipping checkout

A `Checkout` context always computes `subtotal + shipping`, but *how* shipping is priced varies. Each pricing rule is a `ShippingStrategy`. The context never contains `if shipping_mode == ...` branches; swapping the algorithm is `set_shipping(...)`.


In [1]:
from abc import ABC, abstractmethod


# --- Strategy ---

class ShippingStrategy(ABC):
    @abstractmethod
    def cost(self, weight_kg: float) -> float:
        ...


In [2]:
# --- Concrete strategies ---

class StandardShipping(ShippingStrategy):
    def cost(self, weight_kg: float) -> float:
        return 5.0 + 1.25 * weight_kg


class ExpressShipping(ShippingStrategy):
    def cost(self, weight_kg: float) -> float:
        return 12.0 + 2.5 * weight_kg


class OvernightShipping(ShippingStrategy):
    def cost(self, weight_kg: float) -> float:
        return 25.0 + 4.0 * weight_kg


In [3]:
# --- Context ---

class Checkout:
    def __init__(self, shipping: ShippingStrategy) -> None:
        self._shipping = shipping

    def set_shipping(self, shipping: ShippingStrategy) -> None:
        self._shipping = shipping

    def total(self, subtotal: float, weight_kg: float) -> float:
        return round(subtotal + self._shipping.cost(weight_kg), 2)


In [4]:
checkout = Checkout(StandardShipping())
print('Standard:', checkout.total(40.0, 2.0))

checkout.set_shipping(ExpressShipping())
print('Express:', checkout.total(40.0, 2.0))

checkout.set_shipping(OvernightShipping())
print('Overnight:', checkout.total(40.0, 2.0))


Standard: 47.5
Express: 57.0
Overnight: 73.0


`Checkout.total` never mentions Standard, Express, or Overnight. The defining behavior of Strategy is that the algorithm is an object you can replace, and the context keeps working against the same interface.
